In [ ]:
!pip install optbinning


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from optbinning import BinningProcess

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
df_fe = pd.read_csv(
    "../data/processed/final_features_v1.csv"
)

TARGET = "SeriousDlqin2yrs"


X = df_fe.drop(
    columns=[TARGET]
)

y = df_fe[TARGET]


print(X.shape)
print(y.value_counts())

In [ ]:
categorical_features = list(
    X.select_dtypes(
        include=["object","category"]
    ).columns
)


numerical_features = list(
    X.select_dtypes(
        include=["int64","float64"]
    ).columns
)

In [ ]:
binning_process = BinningProcess(
    variable_names=list(X.columns)
)


logistic = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear"
)


woe_pipeline = Pipeline([

    (
        "woe",
        binning_process
    ),

    (
        "model",
        logistic
    )

])

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


results=[]


for train_idx, valid_idx in skf.split(X,y):


    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]


    woe_pipeline.fit(
        X_train,
        y_train
    )


    probs = woe_pipeline.predict_proba(
        X_valid
    )[:,1]


    preds = (
        probs >= 0.5
    ).astype(int)


    results.append({

        "ROC-AUC":
        roc_auc_score(
            y_valid,
            probs
        ),

        "PR-AUC":
        average_precision_score(
            y_valid,
            probs
        ),

        "Precision":
        precision_score(
            y_valid,
            preds
        ),

        "Recall":
        recall_score(
            y_valid,
            preds
        ),

        "F1":
        f1_score(
            y_valid,
            preds
        )

    })


woe_results = pd.DataFrame(results)

woe_results.mean()

### hyper parameteric tunning

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from optbinning import BinningProcess

binning_process = BinningProcess(
    variable_names=list(X.columns)
)

woe_pipeline = Pipeline([
    ("woe", binning_process),
    ("model", LogisticRegression(random_state=42))
])

param_grid = {

    "model__C": [
        0.001,
        0.01,
        0.1,
        1,
        5,
        10,
        20,
        50
    ],

    "model__penalty": [
        "l1",
        "l2"
    ],

    "model__solver": [
        "liblinear"
    ],

    "model__class_weight": [
        None,
        "balanced",
        {0:1,1:2},
        {0:1,1:3},
        {0:1,1:5},
        {0:1,1:7},
        {0:1,1:10}
    ],

    "model__max_iter":[1000]

}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=woe_pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=skf,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

grid.fit(X, y)

In [ ]:
results = (
    pd.DataFrame(grid.cv_results_)
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

top_results = results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__C",
        "param_model__penalty",
        "param_model__class_weight"
    ]
]

top_results.columns = [
    "Rank",
    "Mean_PR_AUC",
    "STD",
    "C",
    "Penalty",
    "Class_Weight"
]

top_results.head(20)

In [ ]:
print("Best Parameters")
print(grid.best_params_)

print("\nBest PR-AUC")
print(grid.best_score_)

In [ ]:
best_woe_pipeline = grid.best_estimator_

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    best_woe_pipeline.fit(
        X_train,
        y_train
    )

    probs = best_woe_pipeline.predict_proba(X_valid)[:, 1]

    preds = (probs >= 0.5).astype(int)

    results.append({

        "Accuracy": accuracy_score(
            y_valid,
            preds
        ),

        "ROC-AUC": roc_auc_score(
            y_valid,
            probs
        ),

        "PR-AUC": average_precision_score(
            y_valid,
            probs
        ),

        "Precision": precision_score(
            y_valid,
            preds,
            zero_division=0
        ),

        "Recall": recall_score(
            y_valid,
            preds
        ),

        "F1": f1_score(
            y_valid,
            preds
        )

    })

final_woe_results = pd.DataFrame(results)

display(final_woe_results)

print("\nAverage Performance\n")
display(final_woe_results.mean().to_frame("Mean"))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.model_selection import StratifiedKFold


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

all_results = []

thresholds = np.arange(
    0.01,
    1.00,
    0.01
)

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]


    best_woe_pipeline.fit(
        X_train,
        y_train
    )

    probs = best_woe_pipeline.predict_proba(
        X_valid
    )[:, 1]


    roc = roc_auc_score(
        y_valid,
        probs
    )

    pr = average_precision_score(
        y_valid,
        probs
    )


    fold_results = []

    for threshold in thresholds:

        preds = (
            probs >= threshold
        ).astype(int)

        precision = precision_score(
            y_valid,
            preds,
            zero_division=0
        )

        recall = recall_score(
            y_valid,
            preds,
            zero_division=0
        )

        f1 = f1_score(
            y_valid,
            preds,
            zero_division=0
        )

        fold_results.append({

            "Threshold": threshold,
            "Precision": precision,
            "Recall": recall,
            "F1": f1

        })


    fold_results = pd.DataFrame(fold_results)


    candidates = fold_results[
        fold_results["Recall"] >= 0.70
    ]


    if len(candidates) == 0:

        continue


    best = candidates.sort_values(
        "Precision",
        ascending=False
    ).iloc[0]


    all_results.append({

        "ROC-AUC": roc,
        "PR-AUC": pr,
        "Threshold": best["Threshold"],
        "Precision": best["Precision"],
        "Recall": best["Recall"],
        "F1": best["F1"]

    })


final_results = pd.DataFrame(all_results)

display(final_results)

print("\nAverage Performance\n")

display(
    final_results.mean().to_frame("Mean")
)